# PointPillars: evaluation on KITTI val

Evaluates a trained checkpoint with the refactored `pointpillars` package and collects everything needed for the README:

1. **Metrics**: BEV / 3D AP_R40 at IoU 0.7 and 0.5, orientation similarity, and how much the score threshold matters.
2. **Heading fix**: the same predictions decoded with the old notebook logic and the fixed logic.
3. **Range breakdown**: AP for 0–20 m, 20–40 m and 40+ m.
4. **Speed**: pillarization, forward pass (FP32 and FP16) and post-processing, in ms and Hz.
5. **Official KITTI metric**: easy / moderate / hard numbers from OpenPCDet's evaluator. This step is best-effort.
6. **Pictures**: PR curves, plus BEV plots of typical frames and of the frames with the most missed cars.

**Runtime → Change runtime type → T4 GPU**, edit the settings cell, then *Run all*. The last cell prints a summary block. Paste that block back to Claude.

## 0. Settings: edit these

In [ ]:
# Where the code comes from: a GitHub repo URL, or a zip of the project on Drive.
REPO     = ""                                                   # e.g. "https://github.com/<you>/pointpillars.git"
CODE_ZIP = "/content/drive/MyDrive/pointpillars_code.zip"        # used when REPO is empty

CKPT  = "/content/drive/MyDrive/pointpillars.pth"   # notebook state_dict, or best_model.pth / last.pth from tools/train.py
KITTI = "/content/kitti"                            # needs training/{velodyne,calib,label_2} + ImageSets/val.txt
OUT   = "/content/eval_out"

SCORE_THRS = [0.05, 0.1, 0.3]   # 0.3 is the config default for demos; AP is usually higher at lower thresholds

## 1. Environment + code

In [ ]:
import os, torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
from google.colab import drive
drive.mount("/content/drive")

if REPO:
    !git clone -q $REPO /content/pointpillars || (cd /content/pointpillars && git pull -q)
else:
    !rm -rf /content/pointpillars && mkdir -p /content/pointpillars && unzip -q -o "$CODE_ZIP" -d /content/pointpillars
%cd /content/pointpillars
!pip -q install -e ".[dev]"
!python -m pytest -q
os.makedirs(OUT, exist_ok=True)
assert os.path.exists(CKPT), f"checkpoint not found: {CKPT}"

## 2. KITTI (skipped if already present; the full download takes about 25 min)

In [ ]:
if not os.path.isdir(f"{KITTI}/training/velodyne") or len(os.listdir(f"{KITTI}/training/velodyne")) < 7481:
    !bash scripts/download_kitti.sh $KITTI
print("velodyne scans:", len(os.listdir(f"{KITTI}/training/velodyne")))

## 3. Load model and val set

In [ ]:
import json, time, sys
import numpy as np, torch
from torch.utils.data import DataLoader
from pointpillars.config import load_config
from pointpillars.anchors import make_anchors
from pointpillars.data import KittiDataset, collate, to_device, to_kitti_lines, read_calib
from pointpillars.models import PointPillars
from pointpillars.utils import load_weights
from pointpillars.eval import ap_r40, compute_metrics
from pointpillars.ops.box_utils import apply_direction, decode, limit_period, nms_bev, iou_bev_rot, iou_3d_rot
from pointpillars.ops.pillarize import pillarize
from pointpillars.postprocess import decode_predictions

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = load_config("configs/car.yaml", {"root": KITTI, "train.num_workers": 2})
ds = KittiDataset(cfg, "val", train=False)
n_ids = sum(1 for l in open(f"{KITTI}/ImageSets/val.txt") if l.strip())
assert len(ds) == n_ids, (f"only {len(ds)}/{n_ids} val frames have a velodyne scan - "
                          "the velodyne folder is incomplete, re-run the KITTI cell")
dl = DataLoader(ds, 4, shuffle=False, num_workers=2, collate_fn=collate)
model = PointPillars(cfg).to(dev).eval()
load_weights(model, CKPT, dev)
anchors = make_anchors(cfg).to(dev)
print(len(ds), "val frames |", len(anchors), "anchors")

## 4. Inference (once, at the lowest threshold)

Each frame is decoded twice:
- **new**: the fixed direction decoding from `apply_direction`.
- **old**: the original notebook's decoding, which used `limit_period(..., 0.5, π)`.

Both use the same NMS, so their BEV footprints match and only the heading differs. Detections for the higher score thresholds are taken from this run by filtering. That gives the same result as rerunning, because NMS never lets a lower-scoring box suppress a higher-scoring one.

In [ ]:
MIN_THR = min(SCORE_THRS)
tc = cfg["test"]

@torch.no_grad()
def decode_new_and_old(preds, thr):
    scores = torch.sigmoid(preds["cls"].float()).squeeze(-1)
    new, old = [], []
    for b in range(scores.shape[0]):
        m = scores[b] > thr
        if not m.any():
            z = np.zeros((0, 8), np.float32); new.append(z); old.append(z); continue
        boxes = decode(preds["box"][b][m].float(), anchors[m])
        dcls = preds["dir"][b][m].argmax(-1)
        raw = boxes[:, 6].clone()
        boxes[:, 6] = apply_direction(raw, dcls, cfg["dir_offset"])
        bx, sc = boxes.cpu().numpy(), scores[b][m].cpu().numpy()
        keep = nms_bev(bx, sc, tc["nms_thr"], tc["max_dets"], tc["nms_pre"])
        o = cfg["dir_offset"]
        old_h = (limit_period(raw - o, 0.5, np.pi) + o + np.pi * dcls.float()).cpu().numpy()
        bo = bx.copy(); bo[:, 6] = old_h
        new.append(np.concatenate([bx[keep], sc[keep][:, None]], 1).astype(np.float32))
        old.append(np.concatenate([bo[keep], sc[keep][:, None]], 1).astype(np.float32))
    return new, old

dets_new, dets_old, gts, fids = [], [], [], []
t0 = time.time()
with torch.no_grad():
    for i, batch in enumerate(dl):
        gts += [g.numpy() for g in batch["gt"]]; fids += batch["fid"]
        preds = model(to_device(batch, dev))
        n, o = decode_new_and_old(preds, MIN_THR)
        dets_new += n; dets_old += o
        if i % 200 == 0: print(f"{i}/{len(dl)}")
print(f"inference done in {time.time() - t0:.0f}s | {sum(len(g) for g in gts)} GT cars | "
      f"{sum(len(d) for d in dets_new)} detections @ score>{MIN_THR}")

filt = lambda D, t: [d[d[:, 7] >= t] for d in D]

## 5. Metrics vs. score threshold

In [ ]:
results = {"frames": len(ds), "gt_cars": int(sum(len(g) for g in gts)), "by_score_thr": {}}
for t in SCORE_THRS:
    m = compute_metrics(filt(dets_new, t), gts)
    results["by_score_thr"][str(t)] = m
    print(f"score >= {t:<5}", m)
best_thr = max(SCORE_THRS, key=lambda t: results["by_score_thr"][str(t)]["bev_ap@0.7"])
print("best threshold for BEV AP@0.7:", best_thr)

## 6. Heading fix: old vs. new decoding (same boxes, same scores)

In [ ]:
hf = {}
for name, D in (("old_notebook_decode", dets_old), ("fixed_decode", dets_new)):
    D = filt(D, best_thr)
    bev, osim = ap_r40(D, gts, 0.7, iou_bev_rot)
    ap3d, _ = ap_r40(D, gts, 0.7, iou_3d_rot)
    hf[name] = {"bev_ap@0.7": round(bev * 100, 2), "3d_ap@0.7": round(ap3d * 100, 2),
                "orientation_sim@0.7": round(osim * 100, 2)}
    print(f"{name:22s}", hf[name])

def flipped_share(D):
    # share of true positives (BEV IoU >= 0.7) whose heading is off by more than 90 degrees
    D = filt(D, best_thr); n = f = 0
    for d, g in zip(D, gts):
        if len(g) == 0: continue
        for b in d:
            ious = [iou_bev_rot(b, gg) for gg in g]; j = int(np.argmax(ious))
            if ious[j] >= 0.7:
                n += 1
                f += abs(limit_period(b[6] - g[j][6], 0.5, 2 * np.pi)) > np.pi / 2
    return round(100 * f / max(n, 1), 2)

for name, D in (("old_notebook_decode", dets_old), ("fixed_decode", dets_new)):
    hf[name]["flipped_%"] = flipped_share(D)
    print(f"{name:22s} heading flipped on {hf[name]['flipped_%']}% of true positives")
results["heading_fix"] = hf

## 7. AP by distance

In [ ]:
def in_band(boxes, lo, hi):
    r = np.hypot(boxes[:, 0], boxes[:, 1]) if len(boxes) else np.zeros(0)
    return (r >= lo) & (r < hi)

results["by_distance"] = {}
D = filt(dets_new, best_thr)
for lo, hi in [(0, 20), (20, 40), (40, 80)]:
    Db = [d[in_band(d, lo, hi)] for d in D]
    Gb = [g[in_band(g, lo, hi)] for g in gts]
    bev, _ = ap_r40(Db, Gb, 0.7, iou_bev_rot)
    ap3d, _ = ap_r40(Db, Gb, 0.7, iou_3d_rot)
    key = f"{lo}-{hi}m"
    results["by_distance"][key] = {"gt_cars": int(sum(len(g) for g in Gb)),
                                   "bev_ap@0.7": round(bev * 100, 2), "3d_ap@0.7": round(ap3d * 100, 2)}
    print(key, results["by_distance"][key])

## 8. PR curves

In [ ]:
import matplotlib.pyplot as plt

def pr_curve(D, G, thr, iou_fn):
    npos = sum(len(g) for g in G)
    flat = sorted([(d[7], fi, d[:7]) for fi, Ds in enumerate(D) for d in Ds], key=lambda x: -x[0])
    matched = [np.zeros(len(g), bool) for g in G]; tp = np.zeros(len(flat))
    for k, (_, fi, b) in enumerate(flat):
        g = G[fi]
        if len(g) == 0: continue
        ious = np.array([iou_fn(b, gg) for gg in g]); j = int(ious.argmax())
        if ious[j] >= thr and not matched[fi][j]: tp[k] = 1; matched[fi][j] = True
    ctp = np.cumsum(tp)
    return ctp / max(npos, 1), ctp / np.arange(1, len(flat) + 1)

D = filt(dets_new, MIN_THR)
fig, ax = plt.subplots(figsize=(6, 4.5))
for thr, fn, name in ((0.7, iou_bev_rot, "BEV @0.7"), (0.5, iou_bev_rot, "BEV @0.5"), (0.7, iou_3d_rot, "3D @0.7")):
    r, p = pr_curve(D, gts, thr, fn)
    ax.plot(r, p, label=name)
ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.set_title("PointPillars, KITTI val, Car"); ax.grid(alpha=.3); ax.legend()
fig.tight_layout(); fig.savefig(f"{OUT}/pr_curves.png", dpi=130); plt.show()

## 9. Speed (batch 1)

In [ ]:
N_FRAMES, WARMUP = 100, 10
sync = torch.cuda.synchronize if dev.type == "cuda" else (lambda: None)
ms = lambda ts: round(1000 * float(np.mean(ts)), 2)

raw = [ds.load(i)[0] for i in range(N_FRAMES)]
t_pil = []
for k, pts in enumerate(raw):
    t0 = time.perf_counter(); pillarize(pts, cfg, np.random.default_rng(k)); t_pil.append(time.perf_counter() - t0)

batches = [to_device(collate([ds[i]]), dev) for i in range(N_FRAMES)]
speed = {"pillarize_cpu_ms": ms(t_pil)}
with torch.no_grad():
    for name, amp in (("forward_fp32_ms", False), ("forward_fp16_ms", True)):
        ts = []
        for k, b in enumerate(batches):
            sync(); t0 = time.perf_counter()
            with torch.autocast(dev.type, enabled=amp):
                preds = model(b)
            sync()
            if k >= WARMUP: ts.append(time.perf_counter() - t0)
        speed[name] = ms(ts)
    for thr in (0.1, 0.3):
        ts = []
        for b in batches[:50]:
            preds = model(b); sync()
            t0 = time.perf_counter(); decode_predictions(preds, anchors, cfg, thr); ts.append(time.perf_counter() - t0)
        speed[f"postprocess_thr{thr}_ms"] = ms(ts)
total = speed["pillarize_cpu_ms"] + speed["forward_fp16_ms"] + speed["postprocess_thr0.3_ms"]
speed["end_to_end_fp16_ms"] = round(total, 2)
speed["end_to_end_fp16_hz"] = round(1000 / total, 1)
speed["gpu"] = torch.cuda.get_device_name(0) if dev.type == "cuda" else "cpu"
results["speed"] = speed
print(json.dumps(speed, indent=2))

## 10. Official KITTI metric (easy / moderate / hard), best-effort

This exports KITTI-format predictions and runs OpenPCDet's `kitti_object_eval_python`, which uses numba. If it fails, everything above still stands. Paste the error together with the summary.

In [ ]:
EXP = f"{OUT}/val_preds"
os.makedirs(EXP, exist_ok=True)
calib_dir = f"{KITTI}/training/calib"
for fid, d in zip(fids, filt(dets_new, MIN_THR)):
    lines = to_kitti_lines(d, read_calib(f"{calib_dir}/{fid}.txt"), cfg["cls_name"])
    with open(f"{EXP}/{fid}.txt", "w") as f:
        f.write("\n".join(lines) + ("\n" if lines else ""))
print("exported", len(fids), "files")

!test -d /content/OpenPCDet || git clone -q --depth 1 https://github.com/open-mmlab/OpenPCDet /content/OpenPCDet
!rm -rf /content/kitti_eval && cp -r /content/OpenPCDet/pcdet/datasets/kitti/kitti_object_eval_python /content/kitti_eval && touch /content/kitti_eval/__init__.py
!pip -q install numba

import traceback
results["official"] = None
try:
    sys.path.insert(0, "/content")
    from kitti_eval import kitti_common as kc
    from kitti_eval import eval as keval
    ids = [int(f) for f in fids]
    dt_annos = kc.get_label_annos(EXP, ids)
    gt_annos = kc.get_label_annos(f"{KITTI}/training/label_2", ids)
    out = keval.get_official_eval_result(gt_annos, dt_annos, ["Car"])
    result_str = out[0] if isinstance(out, tuple) else out
    print(result_str)
    results["official"] = result_str
except Exception:
    err = traceback.format_exc()
    print(err)
    results["official_error"] = err[-1500:]

## 11. BEV plots: typical frames and the frames with the most missed cars

In [ ]:
sys.path.insert(0, "tools")
from visualize import plot_frame
from IPython.display import Image, display

D = filt(dets_new, 0.3)
def n_missed(d, g):
    return sum(1 for gg in g if not any(iou_bev_rot(b, gg) >= 0.7 for b in d))
missed = np.array([n_missed(d, g) for d, g in zip(D, gts)])
results["frames_with_missed_cars@0.3"] = int((missed > 0).sum())
worst = [int(i) for i in np.argsort(-missed)[:3]]
typical = [int(i) for i in np.linspace(0, len(ds) - 1, 3).astype(int)]

os.makedirs(f"{OUT}/bev", exist_ok=True)
for tag, idxs in (("typical", typical), ("worst", worst)):
    for i in idxs:
        pts, _, fid = ds.load(i)
        p = f"{OUT}/bev/{tag}_{fid}.png"
        plot_frame(pts, gts[i], D[i], cfg["pc_range"], f"{fid} ({tag}, {missed[i]} missed)", p)
        display(Image(p, width=600))

## 12. Summary: copy everything printed below back to Claude

In [ ]:
results["checkpoint"] = CKPT
results["best_score_thr"] = best_thr
json.dump(results, open(f"{OUT}/eval_summary.json", "w"), indent=2)
print("=" * 30, "COPY FROM HERE", "=" * 30)
print(json.dumps({k: v for k, v in results.items() if k not in ("official", "official_error")}, indent=2))
if results.get("official"): print(results["official"])
if results.get("official_error"): print("OFFICIAL EVAL ERROR:\n", results["official_error"])
print("=" * 30, "COPY TO HERE", "=" * 32)

!cd $OUT && zip -q -r /content/eval_out.zip eval_summary.json pr_curves.png bev
from google.colab import files; files.download("/content/eval_out.zip")